# LeetCode 3748 — Count Stable Subarrays bằng Python

- Link: https://leetcode.com/problems/count-stable-subarrays/
- Độ khó: **Hard**
- Chủ đề: Mảng con **không giảm**, chia thành các **đoạn chạy** (runs), tổng tiền tố, tìm kiếm nhị phân, đếm mảng con theo điểm kết thúc

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def countStableSubarrays(self, nums: List[int], queries: List[List[int]]) -> List[int]:
```

## 1. Mục tiêu học

1. Nhận ra "không có nghịch thế" ⇔ mảng con **không giảm** ⇔ nằm trọn trong một **đoạn chạy không giảm** cực đại.
2. Đếm mảng con không giảm của một đoạn dài `k`: `k(k+1)/2`.
3. Trả lời truy vấn `[l, r]` trong `O(log n)`: phần đầu (đoạn chạy bị cắt), các đoạn chạy trọn vẹn (tổng tiền tố), phần cuối.
4. Cách nhìn thứ hai: `left[i]` = đầu đoạn chạy chứa `i`; số mảng con ổn định kết thúc tại `i` trong `[l, r]` là `i − max(left[i], l) + 1`.
5. Kiểm chứng bằng vét cạn.

## 2. Đề bài & ví dụ

Mảng con **ổn định** nếu không có cặp `i < j` với `nums[i] > nums[j]`. Với mỗi truy vấn `[l, r]`, đếm số mảng con ổn định nằm trọn trong `nums[l..r]`.

**Ví dụ 1:** `nums = [3,1,2]`, `queries = [[0,1],[1,2],[0,2]]` → `[2,3,4]`.

**Ví dụ 2:** `nums = [2,2]`, `queries = [[0,1],[0,0]]` → `[3,1]`.

**Ràng buộc:** `n, q ≤ 10^5`.

In [1]:
from typing import List
from bisect import bisect_left, bisect_right
import random, time

examples = [
    ([3, 1, 2], [[0, 1], [1, 2], [0, 2]], [2, 3, 4]),
    ([2, 2], [[0, 1], [0, 0]], [3, 1]),
]
for a, qs, exp in examples:
    print(a, qs, "-> kỳ vọng", exp)

[3, 1, 2] [[0, 1], [1, 2], [0, 2]] -> kỳ vọng [2, 3, 4]
[2, 2] [[0, 1], [0, 0]] -> kỳ vọng [3, 1]


## 3. Kiến thức nền

### 3.1. Không nghịch thế ⇔ không giảm

Nếu mảng không giảm thì mọi `i < j` có `nums[i] ≤ nums[j]`. Ngược lại, nếu có một vị trí giảm `nums[t] > nums[t+1]` thì đó chính là một nghịch thế. Vậy chỉ cần kiểm tra các cặp **kề nhau**.

### 3.2. Đoạn chạy không giảm (runs)

Cắt mảng tại mọi vị trí `t` có `nums[t] > nums[t+1]` ⇒ các đoạn chạy cực đại. Mảng con ổn định không thể vắt qua chỗ cắt.

In [2]:
def runs(nums):
    out = []; start = 0
    for i in range(1, len(nums) + 1):
        if i == len(nums) or nums[i] < nums[i - 1]:
            out.append((start, i - 1)); start = i
    return out

print(runs([1, 3, 3, 2, 5, 6, 0, 0, 4]))

[(0, 2), (3, 5), (6, 8)]


### 3.3. Số mảng con của một đoạn dài `k`

`k + (k − 1) + … + 1 = k(k+1)/2` — mọi mảng con của đoạn không giảm đều không giảm.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Truy vấn `[l, r]` cắt dãy đoạn chạy thành:

- một **mẩu đầu** (từ `l` tới hết đoạn chạy chứa `l`, hoặc tới `r`),
- vài đoạn chạy **trọn vẹn** ở giữa,
- một **mẩu cuối** (từ đầu đoạn chạy chứa `r` tới `r`).

Đáp án = `T(mẩu đầu) + Σ T(đoạn giữa) + T(mẩu cuối)` với `T(k) = k(k+1)/2`. Tổng các đoạn giữa lấy bằng tổng tiền tố trên danh sách đoạn chạy; vị trí `l`, `r` trong danh sách tìm bằng `bisect`.

Ví dụ 1: đoạn chạy `[3] | [1,2]`. Truy vấn `[0,2]`: `T(1) + T(2) = 1 + 3 = 4`. ✔

## 5. Cách 1 — Brute Force

In [3]:
def brute(nums, queries):
    ans = []
    for l, r in queries:
        c = 0
        for i in range(l, r + 1):
            for j in range(i, r + 1):
                if j > i and nums[j] < nums[j - 1]:
                    break
                c += 1
        ans.append(c)
    return ans

for a, qs, exp in examples:
    print(brute(a, qs), "kỳ vọng", exp)

[2, 3, 4] kỳ vọng [2, 3, 4]
[3, 1] kỳ vọng [3, 1]


**Nút thắt:** `O(L²)` mỗi truy vấn (có `break` sớm nhưng xấu nhất với mảng không giảm).

## 6. Ý tưởng thuật toán / bước đột phá

Tiền xử lý:

- `starts` = danh sách chỉ số bắt đầu các đoạn chạy (tăng dần).
- `pre[t]` = `Σ_{s<t} T(len(run_s))`.

Truy vấn `[l, r]`:

- `i = bisect_right(starts, l)` = chỉ số đoạn chạy đầu tiên **bắt đầu sau** `l`;
- `j = bisect_right(starts, r) − 1` = đoạn chạy chứa `r`.
- Nếu `i > j`: `l` và `r` cùng một đoạn chạy ⇒ `T(r − l + 1)`.
- Ngược lại: mẩu đầu dài `a = starts[i] − l`, đoạn giữa là các run `i..j−1`, mẩu cuối dài `b = r − starts[j] + 1` ⇒ `T(a) + (pre[j] − pre[i]) + T(b)`.

## 7. Các cách tiếp theo

- **Cách 2:** quét tuyến tính mỗi truy vấn với độ dài đoạn chạy hiện tại — `O(L)` mỗi truy vấn.
- **Cách 3:** đoạn chạy + tổng tiền tố + `bisect` — `O(log n)` mỗi truy vấn (lời giải nộp).
- **Cách 4:** mảng `left[i]` + tổng tiền tố của `i − left[i] + 1` + tìm nhị phân trên `left` — `O(log n)`.

### 7.1. Cách 2 — Quét tuyến tính theo điểm kết thúc

In [4]:
def linear_scan(nums, queries):
    ans = []
    for l, r in queries:
        c = 0; run = 0
        for i in range(l, r + 1):
            run = run + 1 if i > l and nums[i] >= nums[i - 1] else 1
            c += run                                  # số mảng con ổn định kết thúc tại i
        ans.append(c)
    return ans

for a, qs, exp in examples:
    print(linear_scan(a, qs), "kỳ vọng", exp)

[2, 3, 4] kỳ vọng [2, 3, 4]
[3, 1] kỳ vọng [3, 1]


### 7.2. Cách 3 — Đoạn chạy + tổng tiền tố (lời giải chính)

In [5]:
def T(k):
    return k * (k + 1) // 2

def runs_prefix(nums, queries):
    n = len(nums)
    starts = []; pre = [0]; s = 0
    for i in range(n):
        if i == 0 or nums[i] < nums[i - 1]:
            starts.append(i)
    for t, st in enumerate(starts):
        en = starts[t + 1] if t + 1 < len(starts) else n
        pre.append(pre[-1] + T(en - st))
    ans = []
    for l, r in queries:
        i = bisect_right(starts, l)
        j = bisect_right(starts, r) - 1
        if i > j:
            ans.append(T(r - l + 1))
        else:
            ans.append(T(starts[i] - l) + pre[j] - pre[i] + T(r - starts[j] + 1))
    return ans

for a, qs, exp in examples:
    print(runs_prefix(a, qs), "kỳ vọng", exp)

[2, 3, 4] kỳ vọng [2, 3, 4]
[3, 1] kỳ vọng [3, 1]


### 7.3. Cách 4 — Theo điểm kết thúc với `left[i]`

Số mảng con ổn định kết thúc tại `i` và bắt đầu `≥ l` là `i − max(left[i], l) + 1`. Vì `left` **không giảm** theo `i`, tồn tại ngưỡng `p` = chỉ số đầu tiên trong `[l, r]` có `left[p] ≥ l`:

- với `i < p`: `left[i] < l` ⇒ đóng góp `i − l + 1` (cấp số cộng);
- với `i ≥ p`: đóng góp `i − left[i] + 1` = `cnt[i]` ⇒ tổng tiền tố.

In [6]:
def by_endpoint(nums, queries):
    n = len(nums)
    left = [0] * n
    for i in range(1, n):
        left[i] = left[i - 1] if nums[i] >= nums[i - 1] else i
    pc = [0] * (n + 1)
    for i in range(n):
        pc[i + 1] = pc[i] + (i - left[i] + 1)
    ans = []
    for l, r in queries:
        p = bisect_left(left, l, l, r + 1)            # left[p] >= l đầu tiên trong [l, r]
        m = p - l                                     # số i trong [l, p): đóng góp 1..m
        ans.append(T(m) + pc[r + 1] - pc[p])
    return ans

for a, qs, exp in examples:
    print(by_endpoint(a, qs), "kỳ vọng", exp)

[2, 3, 4] kỳ vọng [2, 3, 4]
[3, 1] kỳ vọng [3, 1]


## 8. TRACE từng bước

In [7]:
def trace_runs(nums, queries):
    n = len(nums)
    starts = [i for i in range(n) if i == 0 or nums[i] < nums[i - 1]]
    pre = [0]
    for t, st in enumerate(starts):
        en = starts[t + 1] if t + 1 < len(starts) else n
        pre.append(pre[-1] + T(en - st))
    print("nums =", nums)
    print("starts =", starts, "| pre =", pre)
    for l, r in queries:
        i = bisect_right(starts, l); j = bisect_right(starts, r) - 1
        if i > j:
            print(f"[{l},{r}]: cùng một đoạn chạy -> T({r - l + 1}) = {T(r - l + 1)}")
        else:
            a = starts[i] - l; b = r - starts[j] + 1
            print(f"[{l},{r}]: i={i}, j={j}; mẩu đầu {a} -> {T(a)}, giữa pre[{j}]-pre[{i}] = {pre[j] - pre[i]}, "
                  f"mẩu cuối {b} -> {T(b)}; tổng = {T(a) + pre[j] - pre[i] + T(b)}")

trace_runs([1, 3, 3, 2, 5, 6, 0, 0, 4], [[0, 8], [1, 5], [4, 5], [2, 7]])
print("brute:", brute([1, 3, 3, 2, 5, 6, 0, 0, 4], [[0, 8], [1, 5], [4, 5], [2, 7]]))

nums = [1, 3, 3, 2, 5, 6, 0, 0, 4]
starts = [0, 3, 6] | pre = [0, 6, 12, 18]
[0,8]: i=1, j=2; mẩu đầu 3 -> 6, giữa pre[2]-pre[1] = 6, mẩu cuối 3 -> 6; tổng = 18
[1,5]: i=1, j=1; mẩu đầu 2 -> 3, giữa pre[1]-pre[1] = 0, mẩu cuối 3 -> 6; tổng = 9
[4,5]: cùng một đoạn chạy -> T(2) = 3
[2,7]: i=1, j=2; mẩu đầu 1 -> 1, giữa pre[2]-pre[1] = 6, mẩu cuối 2 -> 3; tổng = 10
brute: [18, 9, 3, 10]


In [8]:
def trace_endpoint(nums, l, r):
    n = len(nums); left = [0] * n
    for i in range(1, n):
        left[i] = left[i - 1] if nums[i] >= nums[i - 1] else i
    print("left =", left)
    for i in range(l, r + 1):
        print(f"  i={i}: kết thúc tại i, bắt đầu từ max(left={left[i]}, l={l}) -> {i - max(left[i], l) + 1} mảng con")

trace_endpoint([1, 3, 3, 2, 5, 6, 0, 0, 4], 1, 5)

left = [0, 0, 0, 3, 3, 3, 6, 6, 6]
  i=1: kết thúc tại i, bắt đầu từ max(left=0, l=1) -> 1 mảng con
  i=2: kết thúc tại i, bắt đầu từ max(left=0, l=1) -> 2 mảng con
  i=3: kết thúc tại i, bắt đầu từ max(left=3, l=1) -> 1 mảng con
  i=4: kết thúc tại i, bắt đầu từ max(left=3, l=1) -> 2 mảng con
  i=5: kết thúc tại i, bắt đầu từ max(left=3, l=1) -> 3 mảng con


## 9. Code nộp LeetCode

In [9]:
from bisect import bisect_right

class Solution:
    def countStableSubarrays(self, nums: List[int], queries: List[List[int]]) -> List[int]:
        n = len(nums)
        starts = [i for i in range(n) if i == 0 or nums[i] < nums[i - 1]]
        pre = [0]
        for t, st in enumerate(starts):
            en = starts[t + 1] if t + 1 < len(starts) else n
            k = en - st
            pre.append(pre[-1] + k * (k + 1) // 2)
        ans = []
        for l, r in queries:
            i = bisect_right(starts, l)
            j = bisect_right(starts, r) - 1
            if i > j:
                k = r - l + 1
                ans.append(k * (k + 1) // 2)
            else:
                a = starts[i] - l
                b = r - starts[j] + 1
                ans.append(a * (a + 1) // 2 + pre[j] - pre[i] + b * (b + 1) // 2)
        return ans

sol = Solution()
for a, qs, exp in examples:
    got = sol.countStableSubarrays(a, qs)
    print(got, "✔" if got == exp else "✘")

[2, 3, 4] ✔
[3, 1] ✔


## 10. Vì sao thuật toán đúng

1. **Đặc trưng:** mảng con ổn định ⇔ không giảm ⇔ không chứa vị trí cắt `t` (`nums[t] > nums[t+1]`) với cả `t` và `t+1` trong mảng con ⇔ nằm trọn trong một đoạn chạy cực đại.
2. **Phân hoạch:** giao của `[l, r]` với các đoạn chạy là: mẩu đầu `[l, starts[i] − 1]`, các run `i..j−1` trọn vẹn, mẩu cuối `[starts[j], r]` (khi `i ≤ j`). Mỗi mẩu là đoạn không giảm và mọi mảng con ổn định trong `[l, r]` nằm trong đúng một mẩu ⇒ tổng `T(·)`.
3. **Chỉ số:** `bisect_right(starts, l)` là run đầu tiên bắt đầu **sau** `l` ⇒ mẩu đầu là `[l, starts[i] − 1]`, luôn khác rỗng vì `starts[i] > l` (nên `a ≥ 1`). `j` là run chứa `r`. Nếu `i > j` thì không có run nào bắt đầu trong `(l, r]` ⇒ `[l, r]` nằm trong một run.
4. **Cách 4:** `left` không giảm vì đầu run của `i+1` là `left[i]` hoặc `i+1`. Ngưỡng `p` tách đúng hai trường hợp của `max(left[i], l)`.

## 11. Phân tích Time / Space Complexity

| Cách | Tiền xử lý | Mỗi truy vấn | Space |
|---|---|---|---|
| 1. Vét cạn | — | `O(L²)` | `O(1)` |
| 2. Quét tuyến tính | — | `O(L)` | `O(1)` |
| 3. Runs + tiền tố | `O(n)` | `O(log n)` | `O(n)` |
| 4. `left[i]` + tiền tố | `O(n)` | `O(log n)` | `O(n)` |

## 12. Bộ test

In [10]:
METHODS = [brute, linear_scan, runs_prefix, by_endpoint, lambda a, q: Solution().countStableSubarrays(a, q)]
for a, qs, exp in examples:
    assert all(f(a, qs) == exp for f in METHODS)
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [11]:
# 12.2 ca biên
assert Solution().countStableSubarrays([5], [[0, 0]]) == [1]
assert Solution().countStableSubarrays([1, 2, 3, 4], [[0, 3], [1, 2]]) == [10, 3]        # một run duy nhất
assert Solution().countStableSubarrays([4, 3, 2, 1], [[0, 3], [1, 2]]) == [4, 2]         # mọi run dài 1
assert Solution().countStableSubarrays([1, 1, 1], [[0, 2]]) == [6]                       # bằng nhau vẫn ổn định
print("Ca biên: OK")

Ca biên: OK


In [12]:
# 12.3 ngẫu nhiên
rng = random.Random(3748)
for _ in range(1000):
    n = rng.randint(1, 15); a = [rng.randint(1, 4) for _ in range(n)]; qs = []
    for _ in range(rng.randint(1, 10)):
        l = rng.randrange(n); qs.append([l, rng.randrange(l, n)])
    res = [f(a, qs) for f in METHODS]
    assert all(r == res[0] for r in res), (a, qs)
print("1000 test ngẫu nhiên: OK")

1000 test ngẫu nhiên: OK


In [13]:
# 12.4 test lớn
rng = random.Random(1)
n = 10**5
a = [rng.randint(1, 10**5) if rng.random() < 0.01 else 0 for _ in range(n)]
for i in range(1, n):
    if a[i] == 0: a[i] = a[i - 1] + rng.randint(0, 2)
a[0] = a[0] or 1
qs = []
for _ in range(10**5):
    l = rng.randrange(n); qs.append([l, rng.randrange(l, n)])
t0 = time.time(); r1 = Solution().countStableSubarrays(a, qs); t1 = time.time()
print("thời gian:", round(t1 - t0, 2), "s | khớp Cách 4:", r1 == by_endpoint(a, qs), "| đáp án lớn nhất:", max(r1))

thời gian: 0.06 s | khớp Cách 4: True | đáp án lớn nhất: 17431475


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Coi phần tử bằng nhau là nghịch thế (dùng `>` thành `≥` khi cắt run)

In [14]:
def wrong_strict(nums, queries):
    ans = []
    for l, r in queries:
        c = 0; run = 0
        for i in range(l, r + 1):
            run = run + 1 if i > l and nums[i] > nums[i - 1] else 1         # SAI: phải là >=
            c += run
        ans.append(c)
    return ans

print("sai:", wrong_strict([2, 2], [[0, 1]]), "| đúng:", Solution().countStableSubarrays([2, 2], [[0, 1]]))

sai: [2] | đúng: [3]


### Lỗi 2 — Dùng đoạn chạy của **toàn mảng** mà không cắt ở `l`, `r`

In [15]:
def wrong_no_clip(nums, queries):
    rs = runs(nums); ans = []
    for l, r in queries:
        ans.append(sum(T(e - s + 1) for s, e in rs if e >= l and s <= r))   # SAI: run bị cắt vẫn tính đủ
    return ans

print("sai:", wrong_no_clip([1, 2, 3, 0], [[1, 3]]), "| đúng:", Solution().countStableSubarrays([1, 2, 3, 0], [[1, 3]]))

sai: [7] | đúng: [4]


### Lỗi 3 — `bisect_left` thay cho `bisect_right` khi `l` trùng đầu một run

Với `i`, dùng `bisect_left` khi `l = starts[t]` chỉ làm mẩu đầu dài 0 — kết quả tình cờ vẫn đúng. Nhưng với **`j`**, khi `r` trùng đầu một run, `bisect_left(starts, r) − 1` trỏ về run **trước** ⇒ mẩu cuối bị tính sai:

In [16]:
def wrong_bisect_j(nums, queries):
    n = len(nums); starts = [i for i in range(n) if i == 0 or nums[i] < nums[i - 1]]
    pre = [0]
    for t, st in enumerate(starts):
        en = starts[t + 1] if t + 1 < len(starts) else n; pre.append(pre[-1] + T(en - st))
    ans = []
    for l, r in queries:
        i = bisect_right(starts, l); j = bisect_left(starts, r) - 1              # SAI
        ans.append(T(r - l + 1) if i > j else T(starts[i] - l) + pre[j] - pre[i] + T(r - starts[j] + 1))
    return ans

print("sai:", wrong_bisect_j([3, 1, 2], [[0, 1]]), "| đúng:", Solution().countStableSubarrays([3, 1, 2], [[0, 1]]))

sai: [3] | đúng: [2]


### Lỗi 4 — Đếm số **đoạn chạy** thay vì số mảng con

Một đoạn chạy dài `k` có `k(k+1)/2` mảng con ổn định, không phải 1 hay `k`.

In [17]:
def wrong_count_runs(nums, queries):
    return [sum(1 for i in range(l, r + 1) if i == l or nums[i] < nums[i - 1]) for l, r in queries]

print("sai:", wrong_count_runs([1, 2, 3], [[0, 2]]), "| đúng:", Solution().countStableSubarrays([1, 2, 3], [[0, 2]]))

sai: [1] | đúng: [6]


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Không nghịch thế / tăng dần / không giảm" trên mảng con ⇒ chỉ cần xét cặp **kề nhau** ⇒ chia thành **đoạn chạy**.
- Đếm mảng con thoả tính chất "đóng dưới mảng con" ⇒ đếm theo **điểm kết thúc** (`i − left[i] + 1`).
- Truy vấn đoạn trên cấu trúc đoạn chạy ⇒ hai mẩu biên + tổng tiền tố phần giữa.
- `left[i]` đơn điệu ⇒ tìm nhị phân ngưỡng.

## 15. Biến thể của bài

1. **Tăng nghiêm ngặt**: đổi điều kiện cắt thành `nums[i] ≤ nums[i−1]`.
2. **Có cập nhật điểm**: cập nhật làm thay đổi tối đa 2 vị trí cắt ⇒ segment tree lưu (độ dài tiền tố run, hậu tố run, tổng bên trong).
3. **Đếm mảng con có ≤ `k` vị trí giảm**: hai con trỏ / `left_k[i]`.
4. **Mảng con dài nhất ổn định trong đoạn**: sparse table trên độ dài run.

In [18]:
# Biến thể 3: số mảng con trong [l, r] có tối đa k "vị trí giảm" — vét cạn và hai con trỏ (toàn mảng)
def count_at_most_k_drops(nums, k):
    res = 0; drops = 0; left = 0
    for i in range(len(nums)):
        if i > 0 and nums[i] < nums[i - 1]: drops += 1
        while drops > k:
            if nums[left + 1] < nums[left]: drops -= 1
            left += 1
        res += i - left + 1
    return res

def brute_k(nums, k):
    n = len(nums); c = 0
    for i in range(n):
        d = 0
        for j in range(i, n):
            if j > i and nums[j] < nums[j - 1]: d += 1
            if d > k: break
            c += 1
    return c

rng = random.Random(15)
for _ in range(400):
    a = [rng.randint(1, 4) for _ in range(rng.randint(1, 12))]; k = rng.randint(0, 3)
    assert count_at_most_k_drops(a, k) == brute_k(a, k)
assert count_at_most_k_drops([3, 1, 2], 0) == Solution().countStableSubarrays([3, 1, 2], [[0, 2]])[0]
print("Biến thể 3 OK")

Biến thể 3 OK


## 16. So sánh các cách làm

| Cách | Ý tưởng | Mỗi truy vấn | Khi nào dùng |
|---|---|---|---|
| 1. Vét cạn | Duyệt mọi mảng con | `O(L²)` | Kiểm chứng |
| 2. Quét tuyến tính | Độ dài run hiện tại | `O(L)` | Ít truy vấn |
| 3. Runs + tiền tố | Hai mẩu biên + giữa | `O(log n)` | **Lời giải chuẩn** |
| 4. `left[i]` | Theo điểm kết thúc | `O(log n)` | Tổng quát hoá dễ |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `nums = [1,2,1,2]`, truy vấn `[0,3]`.

**Bài 2 (tính tay).** `nums = [5,6,6,1,2,9,3]`, truy vấn `[1,5]`.

**Bài 3 (điền TODO).** Viết `count_strictly_increasing(nums, queries)` (tăng nghiêm ngặt).

**Bài 4 (tự viết lại).** Viết lại Cách 4 không dùng `bisect` (tự cài tìm nhị phân).

**Bài 5 (mini challenge).** Với mỗi truy vấn, trả về **độ dài lớn nhất** của mảng con ổn định trong `[l, r]` (gợi ý: hai mẩu biên + max trên các run trọn vẹn bằng sparse table).

In [19]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def count_strictly_increasing(nums, queries):
    # TODO
    pass

In [20]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def longest_stable(nums, queries):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Run: `[1,2] | [1,2]`. (2) `T(2) + T(2)`. (3) `6`.

**Bài 2.** (1) Run toàn mảng: `[5,6,6] | [1,2,9] | [3]`. (2) Cắt `[1,5]`: `[6,6]`, `[1,2,9]`. (3) `3 + 6 = 9`.

**Bài 3.** (1) Cắt khi `nums[i] ≤ nums[i−1]`. (2) Giữ công thức. (3) Đối chiếu vét cạn.

**Bài 4.** (1) `lo = l, hi = r + 1`. (2) Tìm `left[mid] ≥ l` đầu tiên. (3) Như cũ.

**Bài 5.** (1) Mẩu đầu dài `a`, mẩu cuối dài `b`. (2) Max độ dài các run `i..j−1` bằng sparse table. (3) `max(a, b, giữa)`; nếu cùng run: `r − l + 1`.

## 19. Đáp án tham khảo cho bài tập

In [21]:
# Bài 1, 2
assert Solution().countStableSubarrays([1, 2, 1, 2], [[0, 3]]) == [6]
assert Solution().countStableSubarrays([5, 6, 6, 1, 2, 9, 3], [[1, 5]]) == [9] == brute([5, 6, 6, 1, 2, 9, 3], [[1, 5]])
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [22]:
# Bài 3, 4
def count_strictly_increasing_sol(nums, queries):
    n = len(nums); starts = [i for i in range(n) if i == 0 or nums[i] <= nums[i - 1]]
    pre = [0]
    for t, st in enumerate(starts):
        en = starts[t + 1] if t + 1 < len(starts) else n; pre.append(pre[-1] + T(en - st))
    out = []
    for l, r in queries:
        i = bisect_right(starts, l); j = bisect_right(starts, r) - 1
        out.append(T(r - l + 1) if i > j else T(starts[i] - l) + pre[j] - pre[i] + T(r - starts[j] + 1))
    return out

def by_endpoint_manual(nums, queries):
    n = len(nums); left = [0] * n
    for i in range(1, n): left[i] = left[i - 1] if nums[i] >= nums[i - 1] else i
    pc = [0] * (n + 1)
    for i in range(n): pc[i + 1] = pc[i] + i - left[i] + 1
    out = []
    for l, r in queries:
        lo, hi = l, r + 1
        while lo < hi:
            mid = (lo + hi) // 2
            if left[mid] >= l: hi = mid
            else: lo = mid + 1
        out.append(T(lo - l) + pc[r + 1] - pc[lo])
    return out

def brute_strict(nums, queries):
    out = []
    for l, r in queries:
        c = 0
        for i in range(l, r + 1):
            for j in range(i, r + 1):
                if j > i and nums[j] <= nums[j - 1]: break
                c += 1
        out.append(c)
    return out

rng = random.Random(4)
for _ in range(500):
    n = rng.randint(1, 12); a = [rng.randint(1, 4) for _ in range(n)]; qs = []
    for _ in range(6):
        l = rng.randrange(n); qs.append([l, rng.randrange(l, n)])
    assert count_strictly_increasing_sol(a, qs) == brute_strict(a, qs)
    assert by_endpoint_manual(a, qs) == brute(a, qs)
print("Bài 3, 4 OK")

Bài 3, 4 OK


In [23]:
# Bài 5: độ dài lớn nhất
def longest_stable_sol(nums, queries):
    n = len(nums); starts = [i for i in range(n) if i == 0 or nums[i] < nums[i - 1]]
    lens = [(starts[t + 1] if t + 1 < len(starts) else n) - starts[t] for t in range(len(starts))]
    m = len(lens); sp = [lens[:]]; j = 1
    while (1 << j) <= m:
        p = sp[-1]; h = 1 << (j - 1); sp.append([max(p[i], p[i + h]) for i in range(m - (1 << j) + 1)]); j += 1
    def rmq(a, b):
        if a > b: return 0
        t = (b - a + 1).bit_length() - 1
        return max(sp[t][a], sp[t][b - (1 << t) + 1])
    out = []
    for l, r in queries:
        i = bisect_right(starts, l); j2 = bisect_right(starts, r) - 1
        if i > j2: out.append(r - l + 1)
        else: out.append(max(starts[i] - l, r - starts[j2] + 1, rmq(i, j2 - 1)))
    return out

def brute_longest(nums, queries):
    out = []
    for l, r in queries:
        best = run = 0
        for i in range(l, r + 1):
            run = run + 1 if i > l and nums[i] >= nums[i - 1] else 1; best = max(best, run)
        out.append(best)
    return out

rng = random.Random(5)
for _ in range(500):
    n = rng.randint(1, 14); a = [rng.randint(1, 4) for _ in range(n)]; qs = []
    for _ in range(6):
        l = rng.randrange(n); qs.append([l, rng.randrange(l, n)])
    assert longest_stable_sol(a, qs) == brute_longest(a, qs)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
starts = [i for i in range(n) if i == 0 or nums[i] < nums[i-1]]
pre[t+1] = pre[t] + T(len(run t))              # T(k) = k(k+1)//2
i = bisect_right(starts, l); j = bisect_right(starts, r) - 1
ans = T(r-l+1) if i > j else T(starts[i]-l) + pre[j]-pre[i] + T(r-starts[j]+1)
```

**Checklist:**

- [ ] Bằng nhau **không** phải nghịch thế (cắt khi `<`)?
- [ ] Cắt run ở `l` và `r`?
- [ ] `bisect_right` cho cả `i` và `j`?
- [ ] Trường hợp `l, r` cùng một run?
- [ ] Đã so với vét cạn?